# XG-BOOST:

---

## Importing Libraries:

In [1]:
import time

import numpy as np

import pandas as pd

import optuna
from optuna.samplers import TPESampler

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, make_scorer

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

## Importing Dataset:

In [2]:
data = pd.read_csv("email_spam_features.csv")

## Building Base Model:

In [3]:
X = data.drop(columns=['Category'])
y = data['Category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

In [4]:
label = LabelEncoder()

y_train = label.fit_transform(y_train)
y_test = label.transform(y_test)

In [5]:
xg = XGBClassifier()

start_time = time.time()
xg.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 0.81s


In [6]:
y_pred = xg.predict(X_test)

print(f"Accuracy: {(accuracy_score(y_test, y_pred) * 100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

Accuracy: 98.55%

Precision: 96.61%

Recall: 91.20%

F1 Score: 93.83%


In [7]:
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual ham', 'Actual spam'], columns=['Predicted ham', 'Predicted spam'])
matrix_df

,Predicted ham,Predicted spam
Actual ham,903,4
Actual spam,11,114


## Hyper-Parameter Tuning:

In [8]:
def objective(trial):

    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    max_depth = trial.suggest_categorical('max_depth', [3, 4, 5, 6, 7, 8, 9, 10, None])
    learning_rate = trial.suggest_categorical('learning_rate', [0.1, 0.01, 0.001, 0.0001, 0.00001])

    model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,                    
    )

    result = np.mean(cross_val_score(estimator=model, X=X_train, y=y_train, scoring='precision', cv=5))

    return result

study = optuna.create_study(direction='maximize', sampler=TPESampler())
study.optimize(objective, n_trials=10, n_jobs=-1)

print("\nBest Parameters:", study.best_params)

[I 2026-10-26 22:48:14,413] A new study created in memory with name: no-name-2ebfa295-4044-44df-9340-fee4b4258c74
[I 2026-10-26 22:48:46,408] Trial 2 finished with value: 0.0 and parameters: {'n_estimators': 379, 'max_depth': 10, 'learning_rate': 1e-05}. Best is trial 2 with value: 0.0.
[I 2026-10-26 22:49:01,812] Trial 1 finished with value: 0.955818013990369 and parameters: {'n_estimators': 675, 'max_depth': 4, 'learning_rate': 0.1}. Best is trial 1 with value: 0.955818013990369.
[I 2026-10-26 22:49:10,000] Trial 0 finished with value: 0.0 and parameters: {'n_estimators': 702, 'max_depth': 7, 'learning_rate': 0.0001}. Best is trial 1 with value: 0.955818013990369.
[I 2026-10-26 22:49:19,678] Trial 3 finished with value: 0.0 and parameters: {'n_estimators': 865, 'max_depth': 6, 'learning_rate': 1e-05}. Best is trial 1 with value: 0.955818013990369.
[I 2026-10-26 22:49:31,014] Trial 5 finished with value: 0.9674437979880463 and parameters: {'n_estimators': 363, 'max_depth': None, 'lear


Best Parameters: {'n_estimators': 363, 'max_depth': None, 'learning_rate': 0.01}


In [9]:
def objective(trial):

    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    max_depth = trial.suggest_categorical('max_depth', [3, 4, 5, 6, 7, 8, 9, 10, None])
    learning_rate = trial.suggest_categorical('learning_rate', [0.1, 0.01, 0.001, 0.0001, 0.00001])

    model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,                    
    )

    result = np.mean(cross_val_score(estimator=model, X=X_train, y=y_train, scoring='f1', cv=5))

    return result

study = optuna.create_study(direction='maximize', sampler=TPESampler())
study.optimize(objective, n_trials=10, n_jobs=-1)

print("\nBest Parameters:", study.best_params)

[I 2026-10-26 22:50:04,779] A new study created in memory with name: no-name-bd416c6b-8f6e-4785-9099-492245f34e4b
[I 2026-10-26 22:50:45,207] Trial 2 finished with value: 0.0 and parameters: {'n_estimators': 503, 'max_depth': 6, 'learning_rate': 0.001}. Best is trial 2 with value: 0.0.
[I 2026-10-26 22:50:51,669] Trial 1 finished with value: 0.9377669301271118 and parameters: {'n_estimators': 609, 'max_depth': 6, 'learning_rate': 0.1}. Best is trial 1 with value: 0.9377669301271118.
[I 2026-10-26 22:50:56,572] Trial 3 finished with value: 0.9377669301271118 and parameters: {'n_estimators': 678, 'max_depth': 6, 'learning_rate': 0.1}. Best is trial 1 with value: 0.9377669301271118.
[I 2026-10-26 22:51:10,302] Trial 0 finished with value: 0.9395214616619357 and parameters: {'n_estimators': 959, 'max_depth': 3, 'learning_rate': 0.01}. Best is trial 0 with value: 0.9395214616619357.
[I 2026-10-26 22:51:18,251] Trial 4 finished with value: 0.0 and parameters: {'n_estimators': 397, 'max_depth


Best Parameters: {'n_estimators': 959, 'max_depth': 3, 'learning_rate': 0.01}


In [10]:
# Priority -> Precision:

Xg = XGBClassifier(n_estimators=400, max_depth=None, learning_rate=0.01)

start_time = time.time()
Xg.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 2.83s


In [11]:
y_pred = Xg.predict(X_test)

print(f"Accuracy: {(accuracy_score(y_test, y_pred) * 100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

Accuracy: 98.06%

Precision: 97.30%

Recall: 86.40%

F1 Score: 91.53%


In [12]:
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual ham', 'Actual spam'], columns=['Predicted ham', 'Predicted spam'])
matrix_df

,Predicted ham,Predicted spam
Actual ham,904,3
Actual spam,17,108


In [13]:
# Priority -> F1:

XG = XGBClassifier(n_estimators=950, max_depth=3, learning_rate=0.01)

start_time = time.time()
XG.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 5.63s


In [14]:
y_pred = XG.predict(X_test)

print(f"Accuracy: {(accuracy_score(y_test, y_pred) * 100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

Accuracy: 98.26%

Precision: 98.20%

Recall: 87.20%

F1 Score: 92.37%


In [15]:
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual ham', 'Actual spam'], columns=['Predicted ham', 'Predicted spam'])
matrix_df

,Predicted ham,Predicted spam
Actual ham,905,2
Actual spam,16,109


## Cross-Validation:

In [16]:
# Priority -> Precision:

y = label.fit_transform(y)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)

scoring = {
    'accuracy' : 'accuracy',
    'precision' : make_scorer(precision_score, pos_label=1),
    'recall' : make_scorer(recall_score, pos_label=1),
    'f1' : make_scorer(f1_score, pos_label=1)
}

result = cross_validate(
    estimator=XGBClassifier(n_estimators=400, max_depth=None, learning_rate=0.01),
    X=X,
    y=y,
    scoring=scoring,
    cv=cv,
    n_jobs=-1
)

metrics = ['accuracy', 'precision', 'recall', 'f1']

for i in metrics:
    score = result[f'test_{i}']
    print(f"{i.capitalize()}: {np.mean(score)*100:.2f}%\n")

Accuracy: 98.25%

Precision: 95.89%

Recall: 89.86%

F1: 92.75%



In [17]:
# Priority -> F1:

y = label.fit_transform(y)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)

scoring = {
    'accuracy' : 'accuracy',
    'precision' : make_scorer(precision_score, pos_label=1),
    'recall' : make_scorer(recall_score, pos_label=1),
    'f1' : make_scorer(f1_score, pos_label=1)
}

result = cross_validate(
    estimator=XGBClassifier(n_estimators=950, max_depth=3, learning_rate=0.01),
    X=X,
    y=y,
    scoring=scoring,
    cv=cv,
    n_jobs=-1
)

metrics = ['accuracy', 'precision', 'recall', 'f1']

for i in metrics:
    score = result[f'test_{i}']
    print(f"{i.capitalize()}: {np.mean(score)*100:.2f}%\n")

Accuracy: 98.43%

Precision: 95.94%

Recall: 91.26%

F1: 93.52%

